#  Customer 360 - Feature Engineering 

This notebook focuses on performing Feature Engineering for better ML Results.

## Import Libraries

In [1]:
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 


%matplotlib inline 
sns.set_theme(style="whitegrid") 
palette = ['lightgreen' , 'indianred']

## Load Dataset

In [2]:
df = pd.read_csv('../data/processed/customer_360_cleaned.csv')

df_features = df.copy()

## Binning Tenure Months

In [3]:
print(f'Min months : {df_features['TenureMonths'].min()}')
print(f'Max months : {df_features['TenureMonths'].max()}')

Min months : 1
Max months : 72


In [4]:
bins = [1,12,24,36,48,60,72]
labels = ['<= Year','2 Years','3 Years','4 Years','5 Years','6 Years']

df_features['TenureGroups'] = pd.cut(df_features['TenureMonths'] , bins=bins , labels=labels)

df_features[['TenureMonths','TenureGroups']].head()

,TenureMonths,TenureGroups
0,23,2 Years
1,9,<= Year
2,21,2 Years
3,12,<= Year
4,21,2 Years


## Support intensity 

In [5]:
df_features['SupportCallsPerTenure'] = (df_features['SupportCalls6M'] / df_features['TenureMonths']).round(2)
df_features[['SupportCalls6M','TenureMonths','SupportCallsPerTenure']].head(3)

,SupportCalls6M,TenureMonths,SupportCallsPerTenure
0,5,23,0.22
1,1,9,0.11
2,5,21,0.24


#### Insight: 

`SupportCallsPerTenure` measures the customer's support-call activity relative to their tenure.

A higher value indicates that a customer makes more support calls compared with the length of their relationship with the company. This may help identify customers experiencing frequent issues or requiring more support.

## Late payment rate

In [6]:
df_features["LatePaymentRate"] = (df_features["LatePayments12M"] / 12) * 100
df_features[['LatePayments12M','LatePaymentRate']].tail(5)

,LatePayments12M,LatePaymentRate
2995,2,16.666667
2996,0,0.000000
2997,0,0.000000
2998,0,0.000000
2999,1,8.333333


#### Insight: 

`LatePaymentRate` represents the proportion of months in the last 12 months in which the customer made a late payment.

A higher value indicates more frequent late payments and may provide useful information about customer payment behavior and potential churn risk.

## Usage per service

In [7]:
df_features["UsagePerService"] = (df_features["MonthlyUsageGB"] / df_features["NumServices"]).round(2)
df_features[['MonthlyUsageGB','NumServices','UsagePerService']].head(3)

,MonthlyUsageGB,NumServices,UsagePerService
0,317.9,5,63.58
1,359.3,1,359.30
2,176.9,3,58.97


#### Insight:

`UsagePerService` measures the customer's monthly usage relative to the number of services they have.

This feature provides additional context about how intensively customers use their services. Customers with higher values may have greater usage concentrated across fewer services.

## Charge per service

In [8]:
df_features["ChargePerService"] = (df_features["MonthlyChargeEGP"] / df_features["NumServices"]).round(2)
df_features[['MonthlyChargeEGP','NumServices','ChargePerService']].head(3)

,MonthlyChargeEGP,NumServices,ChargePerService
0,580.48,5,116.10
1,434.36,1,434.36
2,391.35,3,130.45


#### Insight: 

`ChargePerService` represents the customer's monthly charge relative to the number of services they have.

This feature helps capture the customer's spending level in relation to their service usage and may provide additional information about differences in customer value and behavior.

## Estimated annual charge

In [9]:
df_features["EstimatedAnnualCharge"] = df_features["MonthlyChargeEGP"] * 12
df_features[['MonthlyChargeEGP','EstimatedAnnualCharge']].head(3)

,MonthlyChargeEGP,EstimatedAnnualCharge
0,580.48,6965.76
1,434.36,5212.32
2,391.35,4696.20


#### Insight:

`EstimatedAnnualCharge` estimates the customer's annual spending based on their current monthly charge.

This feature provides a longer-term view of customer value and can help the model identify differences in the financial value of customers.

## Estimated Total Charge

In [10]:
df_features["EstimatedTotalCharge"] = df_features["MonthlyChargeEGP"] * df_features['TenureMonths']
df_features[['MonthlyChargeEGP','TenureMonths','EstimatedTotalCharge']].head(3)

,MonthlyChargeEGP,TenureMonths,EstimatedTotalCharge
0,580.48,23,13351.04
1,434.36,9,3909.24
2,391.35,21,8218.35


#### Insight: 

`EstimatedTotalCharge` estimates the customer's accumulated spending based on their current monthly charge and their tenure.

It is calculated as:

`MonthlyChargeEGP × TenureMonths`

A higher value indicates a customer with a higher monthly charge, longer tenure, or both.

This feature can provide a useful proxy for the customer's historical financial value to the company. However, it is an **estimate**, not the customer's actual total revenue or payment history.

## High-value customer flag

In [11]:
threshold = df_features['EstimatedAnnualCharge'].median()

df_features['HighValueCustomer'] = (df_features['EstimatedAnnualCharge'] > threshold).astype(int)

print(f'Threshold: {threshold.round(2)}')
df_features[['EstimatedAnnualCharge','HighValueCustomer']].head(3)

Threshold: 5819.94


,EstimatedAnnualCharge,HighValueCustomer
0,6965.76,1
1,5212.32,0
2,4696.20,0


#### Insight:

`HighValueCustomer` identifies customers whose estimated annual charge is above the median annual charge in the dataset.

- **1** → Above the median estimated annual charge
- **0** → At or below the median estimated annual charge

This feature provides a simple way to distinguish customers with relatively higher estimated spending from the rest of the customer base.

## Satisfaction × Support interaction

In [12]:
df_features['SupportSatisfactionInteraction'] = (
    df_features['SupportCalls6M'] * (5 - df_features['SatisfactionScore']))

df_features[['SupportCalls6M','SatisfactionScore','SupportSatisfactionInteraction']].head(3)

,SupportCalls6M,SatisfactionScore,SupportSatisfactionInteraction
0,5,5.0,0.0
1,1,4.0,1.0
2,5,5.0,0.0


#### Insight: 

`SupportSatisfactionInteraction` combines the number of support calls with the customer's dissatisfaction level.

Since `SatisfactionScore` ranges from 1 to 5, `5 - SatisfactionScore` converts the satisfaction score into a dissatisfaction measure:

- A score of **5** → 0 dissatisfaction
- A score of **1** → 4 dissatisfaction

The interaction becomes higher when a customer has **more support calls and lower satisfaction**.

Therefore, a **higher value indicates a potentially more concerning customer experience**, while a lower value indicates fewer support interactions or higher satisfaction.

## Save the engineered dataset

In [13]:
df_features.to_csv("../data/processed/customer_360_features.csv", index=False)

## Feature Engineering Summary

The feature engineering process created additional variables to capture different aspects of customer behavior and value.

The engineered features capture:

- **Customer experience:** `SupportSatisfactionInteraction`
- **Customer value:** `EstimatedAnnualCharge`, `EstimatedTotalCharge`, and `HighValueCustomer`
- **Payment behavior:** `LatePaymentRate`
- **Service usage:** `UsagePerService`
- **Spending behavior:** `ChargePerService`
- **Customer lifecycle:** `TenureGroup`
- **Support intensity:** `SupportCallsPerTenure`

These features provide the machine learning model with additional information beyond the original raw variables and help represent customer behavior, engagement, experience, and value in a more meaningful way.